In [0]:
df_bronze = spark.read.format("delta").load("/Volumes/workspace/default/lakehouse/bronze/nyctaxi")

In [0]:
df_silver = df_bronze
display(df_silver.limit(5))

In [0]:
from pyspark.sql.functions import col

df_silver = (df_bronze
             
    .filter(col("fare_amount") > 0)
    .filter(col("trip_distance") > 0)
    .dropDuplicates()
    .withColumnRenamed("tpep_pickup_datetime", "pickup_ts")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_ts")
    
)

df_silver.printSchema()

In [0]:
display(df_silver.limit(5))

In [0]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_nyctaxi"))

print("Silver gravado com sucesso.")

In [0]:
spark.sql("SELECT COUNT(*) AS total_linhas FROM workspace.default.silver_nyctaxi").show()

In [0]:
from pyspark.sql.functions import date_format

df_silver_particionado = df_silver.withColumn(
    "pickup_month",
    date_format(col("pickup_ts"), "yyyy-MM")
)

df_silver_particionado.select("pickup_ts", "pickup_month").show(5)

In [0]:
(df_silver_particionado.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_month")           # ← esta linha é a novidade
    .saveAsTable("workspace.default.silver_nyctaxi"))

print("Silver gravado com particionamento por mês.")

In [0]:
display(spark.sql("SHOW PARTITIONS workspace.default.silver_nyctaxi"))


In [0]:
df_silver_particionado.filter(col("pickup_month") == "2016-02").explain()

In [0]:
spark.sql("DESCRIBE DETAIL workspace.default.silver_nyctaxi").select("partitionColumns").show(truncate=False)

In [0]:
df_silver_check = spark.read.table("workspace.default.silver_nyctaxi")
df_silver_check.filter(col("pickup_month") == "2016-01").explain()
